# Partitions and Buckets for the Incident Window

**Module 5: Partitioning and Bucketing for Observability Data** · [Lesson](course.md) · [Quiz](quiz05_partition_and_bucket_choices.ipynb)

When the payment failures began, the questions narrowed to a time window and to a few services: the orders since the failures started, the charges of the payment service. A query that reads only the data of that window and of those services finishes sooner, and leaves the machine to everyone else's queries. Doris makes this possible by dividing each table twice: by time into partitions, and each partition into buckets, which Doris stores as tablets.

In this Lab you look at how the exporter divided `otel_traces`, check with `EXPLAIN` which partitions and tablets a query of the incident reads, and compare the exporter's RANDOM bucketing with bucketing by service on a copy of the spans. Every cell can be rerun.

Plan about 35 minutes. You need the DOG stack and the demo from Module 1, with the payment failure turned on (section 5 of Lab 1).

### Initialize the Lab

Run this cell once before using `lab.shell(...)` or `lab.sql(...)`.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)


## 1. Check the incident's spans

The queries of this Lab ask about the incident window: the time from the first failed order until now. Find the first and the latest failed order:

In [ ]:
incident = lab.wait_for_rows(
    """
    SELECT MIN(timestamp) AS first_failed_order,
           MAX(timestamp) AS latest_failed_order,
           COUNT(*) AS failed_orders
    FROM otel_traces
    WHERE service_name = 'checkout'
      AND span_name = 'oteldemo.CheckoutService/PlaceOrder'
      AND status_code = 'STATUS_CODE_ERROR'
    HAVING COUNT(*) > 0
    """,
    title="The failed orders in otel_traces",
    hint="No order has failed. Check that the demo runs (lab.start_demo() in Module 1) and that paymentFailure "
         "is 50% (section 5 of Lab 1), wait a minute or two, then rerun this cell.",
    timeout_seconds=180,
)
INCIDENT_START = str(incident["first_failed_order"][0])
print("The incident window starts at", INCIDENT_START, "UTC")

**Expected result**

One row: the times of the first and the latest failed order in UTC, and how many orders have failed since the first, about one more for each minute since then. The cell keeps the first time in `INCIDENT_START`.

## 2. How the exporter divided `otel_traces`

The exporter created `otel_traces` with `PARTITION BY RANGE(timestamp)` and dynamic partitioning by day, and with `DISTRIBUTED BY RANDOM BUCKETS AUTO`. A **partition** holds the rows of one range of the partition column, here one UTC day of span start times. Each partition is divided into **buckets**, and Doris stores each bucket as a **tablet**, the unit that a BE stores, copies, and scans. List the partitions:

In [ ]:
lab.sql("SHOW PARTITIONS FROM otel_traces", title="The partitions of otel_traces",
        columns=["PartitionName", "Range", "DistributionKey", "Buckets", "VisibleVersion"]);

**Expected result**

Five partitions, named after their day, such as `p20261020`: the three days before today, today, and tomorrow, in UTC. `Range` gives each partition's first and its excluded last time. `DistributionKey` is `RANDOM` and `Buckets` is 10 for every partition. `VisibleVersion` counts the loads: each Stream Load of the exporter adds a version, several a minute, so today's partition has dozens to thousands, and a partition at version 1 has never received a row.

The partitions come from the table's dynamic partitioning, which a scheduler applies every 10 minutes:

In [ ]:
lab.sql("SHOW DYNAMIC PARTITION TABLES FROM otel", title="Dynamic partitioning of the exporter's tables",
        columns=["TableName", "TimeUnit", "Start", "End", "Buckets", "LastSchedulerTime"]);

**Expected result**

Eight rows, one per exporter table, all `DAY`, `-7`, `1`, and `10`. `End` = 1: the scheduler creates the partition of tomorrow in advance, so that the first spans after midnight have a partition to go to. `Start` = -7: it drops the partitions whose day is more than 7 days before today, which deletes a day of telemetry at once, without rewriting any data. The exporter set these from its settings `history_days` and `create_history_days` (Module 2): `create_history_days: 3` is why the table started with the three past days. `LastSchedulerTime` shows when the scheduler last ran, within the last 10 minutes.

`Buckets` is 10 although the table says `BUCKETS AUTO`. The partitions that Doris creates together with the table get `dynamic_partition.buckets`, 10 by default. For each later partition, Doris estimates the number from the size of the previous partitions, about one bucket per 5 GB of compressed data; when the estimate is less than half of the previous partition's number, Doris keeps the previous number. The course's partitions hold megabytes, so the estimate is 1, and every partition keeps 10 buckets.

## 3. Which partitions a query reads

A query that filters on the partition column reads only the partitions whose range can hold matching rows: **partition pruning**. `EXPLAIN` shows the plan of a query without running it. `lab.explain_scan(...)` runs `EXPLAIN` and shows, for each scan of a table, its filters and how many partitions and tablets it reads, out of how many there are; the whole plan is in a collapsed box.

Count the payment service's spans of the incident window, then look at the plan of the same query:

In [ ]:
INCIDENT_QUERY = """
SELECT span_name,
       COUNT(*) AS spans,
       SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) AS failed
FROM {table}
WHERE service_name = 'payment'
  AND timestamp >= '{start}'
GROUP BY span_name
ORDER BY spans DESC
"""
lab.sql(INCIDENT_QUERY.format(table="otel_traces", start=INCIDENT_START),
        title="The payment service's spans since the first failed order")
lab.explain_scan(INCIDENT_QUERY.format(table="otel_traces", start=INCIDENT_START),
                 title="Which partitions and tablets of otel_traces the query reads");

**Expected result**

The query: `grpc.oteldemo.PaymentService/Charge` is the payment service's server span of each charge request, and about half of them failed; `charge` is a span inside it, which does not fail. Rows with a span or two can follow, such as `grpc.flagd.evaluation.v1.Service/ResolveFloat`, in which the payment service asked flagd for the flag's value.

The plan: one scan of `otel.otel_traces` that reads the table itself (`reads` is `otel_traces`), with both filters under `predicates`. `partitions` is `1/5` followed by today's partition: the filter on `timestamp` rules out every other partition. If the incident started before midnight UTC, it is `2/5`, yesterday's and today's. `tablets` is `10/10`: in the partitions that it reads, the scan reads every tablet, although the query asks for one service. Section 4 explains why.

Two more plans show where pruning comes from. The first query asks for yesterday; the second has no filter on time. The cell computes yesterday's range in UTC and writes it into the query as text, so that Doris knows the values when it plans the query:

In [ ]:
from datetime import datetime, timedelta, timezone

today = datetime.now(timezone.utc).date()
lab.explain_scan(f"""
SELECT COUNT(*) AS spans
FROM otel_traces
WHERE timestamp >= '{today - timedelta(days=1)}' AND timestamp < '{today}'
""", title="The plan of a query for yesterday")
lab.explain_scan("""
SELECT span_name, COUNT(*) AS spans
FROM otel_traces
WHERE service_name = 'payment'
GROUP BY span_name
""", title="The plan of a query without a filter on time");

**Expected result**

The query for yesterday has no table scan, and a card says so: only yesterday's partition can hold its rows, and that partition is empty, so Doris answers without reading. (If the demo ran yesterday, as when you kept the data with `lab.teardown(keep_data=True)`, the plan reads that partition.)

The query without a filter on time also reads `1/5` partitions: Doris also skips partitions that hold no rows, and only today's has any. On a table with a week of telemetry, this query would read all eight partitions, and the query of the incident window would still read one or two. In a production table, pruning by time is the first thing that keeps a query small: always filter on the partition column, with values that Doris knows when it plans the query, such as a time that you computed or `UTC_TIMESTAMP() - INTERVAL 1 HOUR`.

## 4. Buckets: RANDOM or HASH

Within a partition, the distribution decides which bucket each row goes to:

- `DISTRIBUTED BY HASH(column)`: by a hash of the column's value. All rows with the same value go to the same bucket, so a filter `column = value` reads one bucket: **bucket pruning**. The buckets are only as even as the values: a value with many rows makes its bucket large.
- `DISTRIBUTED BY RANDOM`: without a column. A load sends each batch of rows to one bucket and the next batch to the next bucket, so the buckets stay about even, but no filter can prune buckets.

The exporter chose RANDOM. Count the rows in each tablet of `otel_traces`. `lab.tablet_rows(...)` lists the tablets of each partition that has data with `SHOW TABLETS` and counts each tablet's rows with `SELECT COUNT(*) FROM ... TABLET(<id>)`:

In [ ]:
lab.tablet_rows("otel_traces");

**Expected result**

The statements first, then ten tablets of today's partition, each with roughly a tenth of its rows: `percent_of_partition` is between about 7 and 13 for every tablet. The shares come closer to 10 as more batches arrive.

Now bucket the same spans by service. The next cell creates `spans_by_service`, a copy of the main columns of `otel_traces`, with `DISTRIBUTED BY HASH(service_name) BUCKETS 10`, and loads every span of `otel_traces` into it. The table is partitioned by day too, with `AUTO PARTITION`, which section 5 explains.

In [ ]:
lab.execute("""
CREATE TABLE IF NOT EXISTS spans_by_service (
    timestamp      DATETIME(6) NOT NULL,
    service_name   VARCHAR(200),
    trace_id       VARCHAR(200),
    span_id        STRING,
    parent_span_id STRING,
    span_name      STRING,
    span_kind      STRING,
    end_time       DATETIME(6),
    duration       BIGINT,
    status_code    STRING,
    status_message STRING
)
DUPLICATE KEY(timestamp, service_name)
AUTO PARTITION BY RANGE (date_trunc(timestamp, 'day')) ()
DISTRIBUTED BY HASH(service_name) BUCKETS 10
PROPERTIES ("partition.retention_count" = "7")
""", title="Create spans_by_service, bucketed by service")
lab.execute("TRUNCATE TABLE spans_by_service", title="Empty the table, so that the cell can be rerun")
lab.execute("""
INSERT INTO spans_by_service
SELECT timestamp, service_name, trace_id, span_id, parent_span_id, span_name, span_kind, end_time, duration, status_code, status_message
FROM otel_traces
""", title="Copy the spans of otel_traces")
lab.tablet_rows("spans_by_service");

**Expected result**

The insert reports one affected row per span, tens of thousands or more. Then ten tablets, far from even. The tablets that hold the busiest services, such as `frontend-proxy` and `frontend`, have a quarter or more of the rows each, one or two tablets have few rows or none, and the payment service's tablet is small. Which services share a tablet depends only on their hashes. A tablet that holds most of a partition makes its loads, its compaction, and every scan of it slower than the others: the partition is only as fast as its largest tablet.

Look at the plan of the incident query on the copy:

In [ ]:
lab.explain_scan(INCIDENT_QUERY.format(table="spans_by_service", start=INCIDENT_START),
                 title="Which partitions and tablets of spans_by_service the query reads");

**Expected result**

One scan of `otel.spans_by_service`, `partitions` `1/1`, and `tablets` `1/10`: with HASH bucketing on `service_name`, Doris computes the bucket of `'payment'` and reads only that tablet.

So bucketing is a trade-off between pruning and balance. Bucket by a column when most queries filter on it with `=` or `IN` and its values spread the rows evenly; a column with many distinct values, none of them dominant, spreads them best. Raw telemetry is mostly read by time and searched in many ways, and its services are very uneven, so the exporter uses RANDOM: even tablets, and other ways to find the rows of one service, such as the inverted index of `service_name` (Module 6).

## 5. AUTO PARTITION: partitions that the data creates

`spans_by_service` is partitioned with `AUTO PARTITION BY RANGE (date_trunc(timestamp, 'day'))`: it was created without partitions, and Doris created a partition for each day that a loaded row belonged to. List them:

In [ ]:
lab.sql("SHOW PARTITIONS FROM spans_by_service", title="The partitions of spans_by_service",
        columns=["PartitionName", "Range", "Buckets", "VisibleVersion"]);

**Expected result**

One partition per day of the copied spans, usually one, today's, named like `p20261020000000`, with 10 buckets. Unlike the dynamic partitions of `otel_traces`, there is no empty partition: no past days that hold nothing, and no partition for tomorrow until a row of tomorrow arrives.

AUTO PARTITION does not drop old partitions by itself. The table's property `"partition.retention_count" = "7"` asks Doris to keep the seven newest partitions before the current day and drop older ones; the same scheduler as for dynamic partitions applies it, every 10 minutes, so it takes effect only once the table has data of more than seven past days. `SHOW CREATE TABLE spans_by_service` shows the property. An AUTO PARTITION table may also have dynamic partitioning, but not together with `partition.retention_count`.

## 6. Exercise: bucket the spans by trace

When an engineer opens a trace, as you did in Module 3, Doris looks up all the spans of one `trace_id`. Create `spans_by_trace`, with the same columns and partitioning as `spans_by_service` but `DISTRIBUTED BY HASH(trace_id) BUCKETS 10`, and load the spans of `otel_traces` into it. How evenly are its rows spread over the tablets? Take the trace ID of the latest failed order, and check with `lab.explain_scan(...)` how many tablets a query for its spans reads.

In [ ]:
# Your code here

The next cell is a reference solution; it is collapsed. Expand it to compare, or run it.

In [ ]:
lab.execute("""
CREATE TABLE IF NOT EXISTS spans_by_trace (
    timestamp      DATETIME(6) NOT NULL,
    service_name   VARCHAR(200),
    trace_id       VARCHAR(200),
    span_id        STRING,
    parent_span_id STRING,
    span_name      STRING,
    span_kind      STRING,
    end_time       DATETIME(6),
    duration       BIGINT,
    status_code    STRING,
    status_message STRING
)
DUPLICATE KEY(timestamp, service_name)
AUTO PARTITION BY RANGE (date_trunc(timestamp, 'day')) ()
DISTRIBUTED BY HASH(trace_id) BUCKETS 10
""", title="Create spans_by_trace, bucketed by trace")
lab.execute("TRUNCATE TABLE spans_by_trace", title="Empty the table, so that the cell can be rerun")
lab.execute("""
INSERT INTO spans_by_trace
SELECT timestamp, service_name, trace_id, span_id, parent_span_id, span_name, span_kind, end_time, duration, status_code, status_message
FROM otel_traces
""", title="Copy the spans of otel_traces")
lab.tablet_rows("spans_by_trace")

failed_trace = lab.sql("""
SELECT trace_id
FROM spans_by_trace
WHERE service_name = 'checkout'
  AND span_name = 'oteldemo.CheckoutService/PlaceOrder'
  AND status_code = 'STATUS_CODE_ERROR'
ORDER BY timestamp DESC
LIMIT 1
""", title="The trace of the latest failed order")["trace_id"][0]
lab.explain_scan(f"SELECT * FROM spans_by_trace WHERE trace_id = '{failed_trace}'",
                 title="Which tablets a lookup of one trace reads");

**Expected result**

Ten tablets with about a tenth of the rows each: trace IDs are random, and no trace holds more than a few dozen spans. The lookup of one trace reads `1/10` tablets, the one that holds every span of that trace. A table that is mostly read trace by trace, such as the table of an application's traces, can therefore bucket by trace ID and have both even tablets and bucket pruning; Litefuse's tables in Module 7 do. The exporter's `otel_traces` is read in more ways, so it stays with RANDOM.

## You are done when

- you can say how many partitions `otel_traces` has, why, and which of them a query of the incident window reads;
- you have seen that a RANDOM table reads every tablet of a partition, and a table bucketed by service only the service's tablet, at the price of uneven tablets;
- you can say which partitions AUTO PARTITION creates, and what `partition.retention_count` and the dynamic partitioning's `start` drop.

The copies are small, and the later modules do not use them. To remove them, run `DROP TABLE IF EXISTS spans_by_service` and `DROP TABLE IF EXISTS spans_by_trace` with `lab.execute(...)` or in a MySQL client.

## Next

Open [Quiz 5: Partitioning and Bucketing for Observability Data](quiz05_partition_and_bucket_choices.ipynb), then continue to [Module 6: Table Properties, Materialized Views, and Indexes](../module06-property-mv-index/course.md).